# Thetis — Treino ProtoNet Few-Shot no Kaggle

Meta-treino episódico do baseline **ProtoNet** com backbone **R(2+1)D-18** sobre o
dataset [THETIS](https://github.com/THETIS-dataset/dataset), no protocolo
5-way K-shot (val/test em 3-way, conforme o split 6/3/3).

Serve **qualquer uma das 5 modalidades** × {1-shot, 5-shot} — escolha em
`MODALITY` e `K_SHOT` na seção 1, que o resto do notebook se ajusta sozinho.

## Antes de rodar — checklist do painel direito (⚙️ *Session options*)

| Opção | Valor |
|---|---|
| **Accelerator** | `GPU T4 x2` ou `GPU P100` (o código usa **uma** GPU só) |
| **Internet** | `On` — obrigatório (clone do repo, `pip`, pesos pré-treinados do torchvision). Exige conta verificada por telefone. |
| **Persistence** | `Files only` (ou *Variables and Files*) — é o que mantém `/kaggle/working` entre sessões e permite retomar o treino |
| **Input** | um Kaggle Dataset com a pasta da modalidade escolhida |

**Dados:** crie um dataset no Kaggle (*Datasets → New Dataset*) subindo o zip da
pasta da modalidade — o Kaggle descompacta sozinho, e o resultado fica em
`/kaggle/input/<slug>/<PASTA>/<classe>/*.avi`. Gere o zip localmente com:

```bash
cd dataset && zip -r -0 VIDEO_RGB.zip VIDEO_RGB       # rgb
# ou VIDEO_Depth / VIDEO_Mask / VIDEO_Skelet2D / VIDEO_Skelet3D
```

| `MODALITY` | Pasta no dataset | Clipes `.avi` |
|---|---|---|
| `rgb` | `VIDEO_RGB` | 1980 |
| `depth` | `VIDEO_Depth` | 1980 |
| `mask` | `VIDEO_Mask` | 1980 |
| `skeleton_2d` | `VIDEO_Skelet2D` | 1216 |
| `skeleton_3d` | `VIDEO_Skelet3D` | 1217 |

O `manifest.csv` **não** precisa ser subido: a seção 4 o regenera a partir da
própria árvore (os splits de classe são determinísticos pela seed, então saem
idênticos aos do repositório).

**Código:** este notebook clona o repositório do GitHub — faça `git push` das
suas mudanças locais antes de rodar aqui.

## Sobre o limite de sessão (leia antes de escolher `EPOCHS`)

A sessão do Kaggle é cortada em ~12 h, e um commit que estoura o tempo **falha
sem salvar o output**. A run 5-way 5-shot no Colab (L4) levou 7h30 para 100
épocas; numa T4/P100 o 5w1s deve ficar em ~10–14 h — ou seja, não cabe numa
sessão só.

Por isso o trainer grava `last.pt` + `training.json` **a cada época** e aceita
`--resume`. O fluxo recomendado é treinar **em blocos**: `EPOCHS=40` na primeira
sessão, `EPOCHS=80` na segunda, `EPOCHS=100` na terceira — a célula de treino
sempre continua da época seguinte à última salva.

## 0. Ambiente: GPU, RAM, disco e internet

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader \
    || echo "SEM GPU — ative em Session options > Accelerator > GPU"

import os, shutil, socket
import torch

print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")

ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
free = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"RAM: {ram:.1f} GB | livre em /kaggle/working: {free:.1f} GB")

try:
    socket.create_connection(("pypi.org", 443), timeout=5)
    print("Internet: ON")
except OSError:
    print("Internet: OFF  <-- ligue em Session options > Internet (sem isso o clone e os pesos pré-treinados falham)")

## 1. Configuração

`EPOCHS` é a **meta acumulada** desta sessão, não um incremento: com `--resume`,
o treino vai da última época salva até `EPOCHS`. Comece com 40.

In [ ]:
REPO_URL = "https://github.com/CauBitten/thetis.git"
REPO_DIR = "/kaggle/working/Thetis"

# ---- Experimento -----------------------------------------------------------
MODALITY = "rgb"   # rgb | depth | mask | skeleton_2d | skeleton_3d
K_SHOT   = 1       # 1 ou 5

#            modalidade  ->  (pasta no dataset, slug do config, linhas no manifesto)
# O ultimo numero e quantas linhas do manifesto tem essa modalidade preenchida.
_MOD = {
    "rgb":         ("VIDEO_RGB",      "rgb",        1980),
    "depth":       ("VIDEO_Depth",    "depth",      1980),
    "mask":        ("VIDEO_Mask",     "mask",       1980),
    "skeleton_2d": ("VIDEO_Skelet2D", "skeleton2d", 1217),
    "skeleton_3d": ("VIDEO_Skelet3D", "skeleton3d", 1217),
}
assert MODALITY in _MOD, f"MODALITY invalida: {MODALITY!r} -- use um de {list(_MOD)}"
assert K_SHOT in (1, 5), f"K_SHOT deve ser 1 ou 5, nao {K_SHOT}"
VIDEO_DIR, _slug, N_AVI_ESPERADO = _MOD[MODALITY]

BASE_CONFIG = f"experiments/configs/protonet_{_slug}_5w{K_SHOT}s.yaml"
RUN_ID      = f"protonet_{_slug}_5w{K_SHOT}s_kaggle"

# Raiz do dataset anexado (a pasta que CONTEM VIDEO_DIR/).
# None = procurar sozinho em /kaggle/input (desce ate 5 niveis).
# Ou fixe o caminho, ex.: "/kaggle/input/datasets/caubittencourt/video-rgb"
# (pode colar ate .../VIDEO_RGB -- o notebook usa a pasta acima).
DATASET_ROOT = None

WORK_ROOT   = "/kaggle/working"
OUTPUT_ROOT = f"{WORK_ROOT}/outputs"   # checkpoints + resultados da avaliacao
LOG_ROOT    = f"{WORK_ROOT}/logs"      # training.json
DATA_ROOT   = f"{WORK_ROOT}/data"      # manifest gerado aqui

EPOCHS = 40             # meta ACUMULADA desta sessao (40 -> 80 -> 100 em sessoes seguintes)
GRAD_CHECKPOINT = True  # False = ~25-30% mais rapido, porem ~3x mais VRAM de ativacao
                        # (nao altera o resultado, so o uso de memoria)

print(f"{MODALITY} / {K_SHOT}-shot -> {BASE_CONFIG}")
print(f"pasta esperada no dataset: {VIDEO_DIR}/  ({N_AVI_ESPERADO} clipes)")
print(f"run_id: {RUN_ID}")

## 2. Clonar o repositório e instalar dependências

In [ ]:
import os, subprocess
from pathlib import Path

if Path(REPO_DIR, ".git").is_dir():
    # sincroniza com o remoto sem quebrar em clone raso (não apaga arquivos não versionados)
    !git -C {REPO_DIR} fetch --depth 1 origin main && git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

# Este notebook depende do --resume no trainer; se o clone não tiver, é push faltando.
help_txt = subprocess.run(
    ["python", "src/training/meta_trainer.py", "--help"], capture_output=True, text=True
).stdout
assert "--resume" in help_txt, (
    "O repositório clonado não tem --resume em src/training/meta_trainer.py. "
    "Faça commit + push dessa mudança e rode esta célula de novo."
)
print("OK — trainer com --resume")

In [ ]:
# A imagem do Kaggle já traz torch(CUDA)/torchvision/numpy/pandas/opencv/scipy/matplotlib/pyyaml.
# Falta só o decord (decode de vídeo rápido; o loader cai em OpenCV se ele não existir).
!pip install -q decord || echo "decord não instalou — segue com OpenCV"

import importlib
faltando = []
for m in ["torch", "torchvision", "cv2", "pandas", "numpy", "yaml", "scipy", "matplotlib"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        faltando.append((m, str(e)))
print("OK — dependências obrigatórias presentes" if not faltando else f"FALTANDO: {faltando}")

## 3. Localizar os dados

Procura a pasta que contém `{VIDEO_DIR}/` (definida por `MODALITY` na seção 1)
entre os datasets anexados e confere se o `decord` realmente decodifica um clipe
(se não, ele é removido e o loader usa o OpenCV — mais lento apenas na 1ª época,
já que o decode é cacheado em RAM).

In [ ]:
from pathlib import Path

def _find_modality_root(video_dir, explicit=None, base="/kaggle/input", max_depth=5):
    """Retorna a pasta que CONTEM video_dir/, varrendo /kaggle/input em largura.

    O Kaggle monta o dataset em profundidades diferentes conforme como ele foi
    anexado (/kaggle/input/<slug>/ ou /kaggle/input/datasets/<user>/<slug>/),
    entao vale descer alguns niveis em vez de assumir um layout.
    """
    if explicit:
        p = Path(explicit)
        if p.name == video_dir:            # apontaram para dentro da pasta
            p = p.parent
        return p if Path(p, video_dir).is_dir() else None

    base = Path(base)
    if not base.is_dir():
        return None
    nivel = [base]
    for _ in range(max_depth):
        proximo = []
        for d in nivel:
            try:
                filhos = sorted(p for p in d.iterdir() if p.is_dir())
            except PermissionError:
                continue
            if any(p.name == video_dir for p in filhos):
                return d
            proximo.extend(p for p in filhos if p.name != video_dir)
        nivel = proximo
        if not nivel:
            break
    return None

DATASET_ROOT = _find_modality_root(VIDEO_DIR, DATASET_ROOT)
if DATASET_ROOT is None:
    visto = sorted(str(p) for p in Path("/kaggle/input").glob("*")) if Path("/kaggle/input").is_dir() else []
    raise AssertionError(
        f"Nao achei nenhuma pasta {VIDEO_DIR} em /kaggle/input.\n"
        f"O que existe em /kaggle/input: {visto}\n"
        f"Anexe o dataset da modalidade '{MODALITY}' em '+ Add Input' ou preencha "
        f"DATASET_ROOT na secao 1 (pode colar o caminho ate {VIDEO_DIR} -- o notebook usa a pasta acima)."
    )

n_avi = sum(1 for _ in Path(DATASET_ROOT, VIDEO_DIR).rglob("*.avi"))
print(f"DATASET_ROOT = {DATASET_ROOT}  |  {n_avi} videos .avi em {VIDEO_DIR}/")
if n_avi < N_AVI_ESPERADO:
    print(f"AVISO: esperado {N_AVI_ESPERADO} clipes para '{MODALITY}'; achei {n_avi}. Upload incompleto?")

In [ ]:
sample = next(Path(DATASET_ROOT, VIDEO_DIR).rglob("*.avi"))
try:
    import decord
    decord.VideoReader(str(sample))
    print("decord OK -- decode rapido na 1a epoca")
except Exception as e:
    print(f"decord indisponivel ({type(e).__name__}: {e}) -- removendo para o loader cair no OpenCV")

## 4. Gerar o `manifest.csv` e a lista de exclusões

Varre a árvore e monta o manifesto. As colunas das modalidades que você não
anexou ficam vazias — o treino usa só `path_<MODALITY>`, e o sampler descarta as
linhas onde ela está vazia.

O `--full-integrity` é o que gera o `excluded_clips.json`, que remove do pool os
clipes defeituosos que o THETIS traz: vídeos totalmente em branco (a segmentação
do Kinect falha e grava máscara preta) e duplicatas byte a byte (repetições
inexistentes preenchidas com cópia de outro take, o que deixaria o mesmo clipe no
suporte **e** na consulta do mesmo episódio). As checagens rodam só sobre a
modalidade anexada, e o resultado é idêntico ao do dataset completo. Custa alguns
segundos — **sem ele o treino roda sem filtro** e avisa no log.

In [ ]:
!python src/data/loader.py --input "{DATASET_ROOT}" --output "{DATA_ROOT}" --seed 42 --full-integrity

import json
from pathlib import Path

import pandas as pd
MANIFEST_PATH = f"{DATA_ROOT}/processed/manifest.csv"
_df = pd.read_csv(MANIFEST_PATH)
_col = f"path_{MODALITY}"
_com = (_df[_col].fillna("") != "").sum()
print(f"\nmanifest: {len(_df)} linhas | {_com} com {MODALITY} "
      f"| {_df['action_label'].nunique()} classes")
assert _com >= N_AVI_ESPERADO, (
    f"so {_com} linhas tem {_col} preenchido (esperado {N_AVI_ESPERADO}). "
    f"O dataset anexado tem a pasta {VIDEO_DIR}/ completa?"
)

_exc = json.loads(Path(f"{DATA_ROOT}/processed/excluded_clips.json").read_text())
_n = _exc["counts"][MODALITY]
print(f"exclusoes para {MODALITY}: {_n} clipes ({_com - _n} sobram no pool)")
assert _n > 0 or MODALITY not in ("rgb", "depth", "mask", "skeleton_2d", "skeleton_3d"), (
    "lista de exclusao vazia -- o --full-integrity rodou?"
)

## 5. Config derivada para o Kaggle

Parte do config da modalidade escolhida e ajusta **só o que é específico da
máquina**: caminhos dos dados, número de épocas e destinos de checkpoint/log.

> `encoder.batch_size` **não** é mais escalado pela VRAM. O `ProtoNet._encode`
> fatia o lote em chunks desse tamanho e o R(2+1)D-18 tem 37 `BatchNorm3d` que,
> em modo treino, normalizam por chunk — mudar o valor muda o resultado, não só a
> memória, e duas modalidades treinadas com valores diferentes deixam de ser
> comparáveis. Ele vem do config (16, igual em todas). Se faltar VRAM, use
> `GRAD_CHECKPOINT = True`, que reduz memória sem alterar o resultado. Detalhes
> em `experiments/configs/README.md`.

In [ ]:
import yaml, torch
from pathlib import Path

cfg = yaml.safe_load(Path(BASE_CONFIG).read_text())

cfg["data"]["manifest_path"] = str(Path(MANIFEST_PATH).resolve())
cfg["data"]["dataset_root"]  = str(Path(DATASET_ROOT).resolve())

cfg["encoder"]["gradient_checkpointing"] = bool(GRAD_CHECKPOINT)
cfg["optim"]["epochs"] = int(EPOCHS)
cfg["output_root"] = OUTPUT_ROOT
cfg["log_root"]    = LOG_ROOT
cfg["run_id"]      = RUN_ID

KAGGLE_CONFIG = "experiments/configs/_kaggle_active.yaml"
Path(KAGGLE_CONFIG).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
bs = cfg["encoder"]["batch_size"]
assert cfg["modalities"] == [MODALITY], (
    f"config {BASE_CONFIG} tem modalities={cfg['modalities']}, esperado ['{MODALITY}']"
)
print(f"VRAM={vram:.1f} GB | encoder.batch_size={bs} (do config, NAO auto-escalado) | epochs={EPOCHS}")
if vram and vram < 12:
    print(f"  AVISO: batch_size={bs} pode dar OOM em {vram:.1f} GB. Prefira uma GPU maior a")
    print("  baixar o valor -- baixar muda o regime de BatchNorm e invalida a comparacao.")
print()
print(Path(KAGGLE_CONFIG).read_text())

## 6. Retomar de uma sessão anterior (opcional)

Com **Persistence: Files** ligada, `/kaggle/working` já volta pronto e não há
nada a fazer aqui. Se você em vez disso salvou uma versão e anexou o *output*
daquele notebook como input, esta célula copia o `last.pt` de lá para o diretório
de trabalho.

In [ ]:
import shutil, torch
from pathlib import Path

CKPT_DIR = Path(OUTPUT_ROOT, "checkpoints", RUN_ID)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
last = CKPT_DIR / "last.pt"

if not last.exists() and Path("/kaggle/input").is_dir():
    achados = sorted(Path("/kaggle/input").glob(f"**/checkpoints/{RUN_ID}/last.pt"))
    if achados:
        shutil.copy2(achados[0], last)
        print(f"copiado de {achados[0]}")

if last.exists():
    st = torch.load(last, map_location="cpu", weights_only=False)
    lg = st.get("log", {})
    print(f"checkpoint encontrado: época {st['epoch']}/{EPOCHS} "
          f"| best_val={lg.get('best_val_acc')} @ep{lg.get('best_epoch')}")
else:
    print("nenhum checkpoint anterior — o treino começa da época 1")

## 7. Teste de sanidade (smoke)

Roda o pipeline ponta-a-ponta em segundos (encoder aleatório, 1 época, dims
mínimas). Pega erro de caminho/dados **antes** do treino longo. Não mexe no
checkpoint do treino de verdade (o `run_id` vira `smoke_...`).

In [ ]:
!python src/training/meta_trainer.py --config {KAGGLE_CONFIG} --smoke

## 8. Treino

`--resume` continua da época seguinte à do `last.pt` (e começa do zero se não
houver nenhum). A cada época o trainer regrava `last.pt` e `training.json`, e
salva `best.pt` sempre que a acurácia de validação melhora — então uma sessão
interrompida custa no máximo uma época.

A 1ª época é a mais lenta: cada clipe é decodificado uma vez e vai para o cache
em RAM. O tempo de cada época aparece no fim da linha de resumo (`... | 250s`) —
use isso para dimensionar o `EPOCHS` da próxima sessão.

> Para o treino longo, prefira **Save Version → Save & Run All (Commit)**: a
> sessão interativa cai se você fechar a aba, o commit roda em background.
> Só não deixe a meta passar do que cabe em ~12 h — um commit que estoura o
> tempo falha **sem** salvar o output.

Dica: `THETIS_MEM_PROFILE=1` antes do comando imprime o pico de VRAM dos 3
primeiros episódios — se sobrar bastante folga, `GRAD_CHECKPOINT = False` dá
~25-30% de velocidade.

In [ ]:
!python src/training/meta_trainer.py --config {KAGGLE_CONFIG} --resume

## 9. Avaliação no meta_test (1000 episódios)

Roda sobre o `best.pt`. Se o treino ainda não chegou na meta de épocas, isso
continua válido — é só a avaliação do melhor checkpoint até agora.

São 3-way × (K support + 15 query) × 1000 episódios: ~48 mil forwards de clipe no
1-shot, ~60 mil no 5-shot — dezenas de minutos numa T4. Para uma prévia rápida,
acrescente `--n-episodes 200` ao comando.

In [ ]:
from pathlib import Path
ckpt = f"{OUTPUT_ROOT}/checkpoints/{RUN_ID}/best.pt"
assert Path(ckpt).exists(), f"checkpoint não encontrado: {ckpt} (o treino chegou a validar?)"
!python src/training/eval_episodic.py --checkpoint "{ckpt}" --output-root "{OUTPUT_ROOT}"

## 10. Curvas de treino (acurácia + loss) e matriz de confusão

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

log = json.loads(Path(f"{LOG_ROOT}/{RUN_ID}/training.json").read_text())
E = log["epochs"]
ep        = [e["epoch"] for e in E]
train_acc = [e["train_acc"] for e in E]
train_los = [e["train_loss"] for e in E]
# val_acc e val_loss só existem nas épocas de eval; mantidos alinhados por época
val = [(e["epoch"], e.get("val_acc"), e.get("val_loss")) for e in E if "val_acc" in e]

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))

ax_acc.plot(ep, train_acc, label="train_acc", alpha=.85)
if val:
    ax_acc.plot([x for x, _, _ in val], [a for _, a, _ in val], "o-", label="val_acc")
best = log.get("best_epoch")
if best is not None:
    ax_acc.axvline(best, ls="--", c="gray", alpha=.6, label=f"best @ep{best}")
ax_acc.set_xlabel("época"); ax_acc.set_ylabel("acurácia"); ax_acc.grid(alpha=.3); ax_acc.legend()

ax_loss.plot(ep, train_los, label="train_loss", alpha=.85)
vl = [(x, l) for x, _, l in val if l is not None]
if vl:
    ax_loss.plot([x for x, _ in vl], [l for _, l in vl], "o-", label="val_loss")
ax_loss.set_xlabel("época"); ax_loss.set_ylabel("loss"); ax_loss.grid(alpha=.3); ax_loss.legend()

fig.suptitle(f"{RUN_ID} — best_val_acc={log.get('best_val_acc')} @ época {log.get('best_epoch')}")
fig.tight_layout(); plt.show()

secs = [e["seconds"] for e in E if "seconds" in e]
if secs:
    print(f"{len(ep)} épocas registradas | média {sum(secs)/len(secs):.0f}s/época "
          f"| ~{sum(secs)/3600:.1f}h de GPU acumuladas")

In [ ]:
from IPython.display import Image, display
from pathlib import Path
png = f"{OUTPUT_ROOT}/results/{RUN_ID}/confusion.png"
display(Image(filename=png)) if Path(png).exists() else print("confusion.png ainda não gerado")

## 11. Continuar numa próxima sessão

**Com `Persistence: Files only`** (mais simples): reabra o notebook, aumente
`EPOCHS` na seção 1 e rode tudo — `/kaggle/working` voltou com os checkpoints e
a seção 8 retoma sozinha.

**Sem persistence** (ou entre notebooks diferentes):

1. *Save Version → Save & Run All (Commit)* nesta sessão;
2. na próxima, **+ Add Input → Your Work →** o output desta versão;
3. aumente `EPOCHS` e rode tudo: a seção 6 acha o `last.pt` no input e copia.

**Baixar os pesos:** painel *Output* → `outputs/checkpoints/<run_id>/best.pt`
(~360 MB) e `logs/<run_id>/training.json`.

Para trazer os resultados de volta ao repositório, commite o `training.json` e o
`outputs/results/<run_id>/metrics.json` — são leves e versionados.